In [9]:
import lmetk
import os

print(os.environ['PROJECT_ROOT'])  # should print the detected root

import hydra
from lmetk.data.utils import reconstruct_datetime_from_features
import numpy as np
from lmetk.data.datasets import SWDataset, SWBatchDataset

%load_ext autoreload
%autoreload 2

/home/levtel/projects/TabPHM
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Refactored version

In [ ]:
data_name = "ETTh1"

with hydra.initialize(version_base="1.3", config_path="../configs", job_name="run"):
    # print('Current config file: ', config_file)
    parameters = hydra.compose(
        config_name="run.yaml",
        overrides=[f"dataset={data_name}"],
        return_hydra_config=True,
    )
    dataset_loader2 = hydra.utils.instantiate(parameters.dataset.loader)
    dataset_loader2._pipeline()


# Get params of splitting and sliding
data_splitter_params = parameters.dataset.loader.data_splitter
seq_len = data_splitter_params.seq_len
label_len = data_splitter_params.label_len
pred_len = data_splitter_params.pred_len

# get data
splitted_data, splitted_timestamps = dataset_loader2.get_data()

dataset_train = SWDataset(
    features=splitted_data["train"],
    time_features=splitted_timestamps["train"],
    seq_len=seq_len,
    label_len=label_len,
    pred_len=pred_len,
)

dataset_val = SWDataset(
    features=splitted_data["val"],
    time_features=splitted_timestamps["val"],
    seq_len=seq_len,
    label_len=label_len,
    pred_len=pred_len,
)
dataset_test = SWDataset(
    features=splitted_data["test"],
    time_features=splitted_timestamps["test"],
    seq_len=seq_len,
    label_len=label_len,
    pred_len=pred_len,
)
print(f"Train dataset size: {dataset_train.__len__()}")
print(f"Validation dataset size: {dataset_val.__len__()}")
print(f"Test dataset size: {dataset_test.__len__()}")

Train dataset size: 8161
Validation dataset size: 2785
Test dataset size: 5805


In [5]:
for i in range(2783, 2787):
    seq_x, seq_y, seq_x_mark, seq_y_mark = dataset_val.__getitem__(i)
    print("Index", i)
    print(
        f"seq_x shape: {seq_x.shape}, seq_y shape: {seq_y.shape}, seq_x_mark shape: {seq_x_mark.shape}, seq_y_mark shape: {seq_y_mark.shape}"
    )

Index 2783
seq_x shape: (384, 1), seq_y shape: (192, 1), seq_x_mark shape: (384, 4), seq_y_mark shape: (192, 4)
Index 2784
seq_x shape: (384, 1), seq_y shape: (192, 1), seq_x_mark shape: (384, 4), seq_y_mark shape: (192, 4)


AssertionError: Target sequence indices out of bounds

In [7]:
for mode in ["val"]:
    batch_dataset = SWBatchDataset(
        features=splitted_data[mode],
        time_features=splitted_timestamps[mode],
        seq_len=seq_len,
        label_len=label_len,
        pred_len=pred_len,
        stride=1,
    )
    print(f" {mode} size: {batch_dataset.__len__()}")
    for i in range(batch_dataset.__len__()):
        batch_dataset.__getitem__(i)
    seq_x2, seq_y2, seq_x_mark2, seq_y_mark2 = batch_dataset.__getitem__(i)


seq_x2, seq_y2, seq_x_mark2, seq_y_mark2 = batch_dataset.__getitem__(4)
seq_x1, seq_y1, seq_x_mark1, seq_y_mark1 = dataset_val.__getitem__(4)


np.testing.assert_array_almost_equal(seq_x2, seq_x1)

 val size: 2785


### Railway: classical dataset 

In [3]:
import hydra

data_name = "railway"

with hydra.initialize(version_base="1.3", config_path="../configs", job_name="run"):
    # print('Current config file: ', config_file)
    parameters = hydra.compose(
        config_name="run.yaml",
        overrides=[f"dataset={data_name}"],
        return_hydra_config=True,
    )
    dataset_loader2 = hydra.utils.instantiate(parameters.dataset.loader)
    dataset_loader2._pipeline()

In [4]:
data_splitter_params = parameters.dataset.loader.data_splitter
seq_len = data_splitter_params.seq_len
label_len = data_splitter_params.label_len
pred_len = data_splitter_params.pred_len

# get data
splitted_data, splitted_timestamps = dataset_loader2.get_data()

dataset_train = SWDataset(
    features=splitted_data["train"],
    time_features=splitted_timestamps["train"],
    seq_len=seq_len,
    label_len=label_len,
    pred_len=pred_len,
)

dataset_val = SWDataset(
    features=splitted_data["val"],
    time_features=splitted_timestamps["val"],
    seq_len=seq_len,
    label_len=label_len,
    pred_len=pred_len,
)
dataset_test = SWDataset(
    features=splitted_data["test"],
    time_features=splitted_timestamps["test"],
    seq_len=seq_len,
    label_len=label_len,
    pred_len=pred_len,
)
print(f"Train dataset size: {dataset_train.__len__()}")
print(f"Validation dataset size: {dataset_val.__len__()}")
print(f"Test dataset size: {dataset_test.__len__()}")

Train dataset size: 37921
Validation dataset size: 4201
Test dataset size: 8185


In [5]:
# Test should stard from "2023-01-24 00:00:00" (if apply_lookback == False then __getitem__(seq_len) otherwise __getitem__(0))
seq_x, seq_y, seq_x_mark, seq_y_mark = dataset_test.__getitem__(seq_len)
reconstruct_datetime_from_features(
    seq_x_mark.numpy(), ["year", "month", "day", "weekday", "hour"]
)[:3]

0   2023-01-24 00:00:00
1   2023-01-24 01:00:00
2   2023-01-24 02:00:00
dtype: datetime64[ns]

In [6]:
str(
    reconstruct_datetime_from_features(
        seq_x_mark.numpy(), ["year", "month", "day", "weekday", "hour"]
    )[0]
) == "2023-01-24 00:00:00"

True

In [7]:
seq_x, seq_y, seq_x_mark, seq_y_mark = dataset_val.__getitem__(seq_len)
reconstruct_datetime_from_features(
    seq_x_mark.numpy(), ["year", "month", "day", "weekday", "hour"]
)[:3]

0   2022-08-02 00:00:00
1   2022-08-02 01:00:00
2   2022-08-02 02:00:00
dtype: datetime64[ns]

### Railway efficient dataset

In [1]:
import hydra
from lmetk.data.utils import reconstruct_datetime_from_features

%load_ext autoreload
%autoreload 2
data_name = "railway"

with hydra.initialize(version_base="1.3", config_path="../configs", job_name="run"):
    # print('Current config file: ', config_file)
    parameters = hydra.compose(
        config_name="run.yaml",
        overrides=[f"dataset={data_name}"],
        return_hydra_config=True,
    )
    dataset_loader2 = hydra.utils.instantiate(parameters.dataset.loader)
    dataset_loader2._pipeline()

data_splitter_params = parameters.dataset.loader.data_splitter
seq_len = data_splitter_params.seq_len
label_len = data_splitter_params.label_len
pred_len = data_splitter_params.pred_len

# get data
splitted_data, splitted_timestamps = dataset_loader2.get_data()

In [2]:
parameters.dataset.loader

{'_target_': 'lmetk.data.datasources.RailwayLoader', 'load_arguments': {'data_name': 'railway', 'data_folder': 'railway', 'data_path': '${paths.data_dir}/${dataset.loader.load_arguments.data_folder}/${dataset.loader.load_arguments.data_name}.csv', 'samples_per_day': 24, 'target_name': 'Bahnlast', 'timestamp_name': 'datetime', 'trend_decomp': False, 'trend_decomp_rolling_window': None, 'time_features': ['year', 'month', 'day', 'weekday', 'hour'], 'normalize_time_features': False}, 'data_splitter': {'_target_': 'lmetk.data.preprocessing.splits.DateBasedSplitter', 'test_start': '2023-01-24 00:00:00', 'train_ratio': 0.9, 'val_ratio': 0.1, 'seq_len': 48, 'label_len': 24, 'pred_len': 24, 'apply_lookback': True}, 'scaler': {'features': {'_target_': 'sklearn.preprocessing.MinMaxScaler'}}}

In [3]:
from lmetk.data.datasets.context_dataset import (
    ContextBatchDataset,
)

for mode in ["train", "val", "test"]:
    context_dataset = ContextBatchDataset(
        features=splitted_data[mode],
        time_features=splitted_timestamps[mode],
        seq_len=seq_len,
        label_len=label_len,
        pred_len=pred_len,
        mask=dataset_loader2.data_dict["mask"],
        stride=1,
    )
    print(f"ContextBatchDataset {mode} size: {context_dataset.__len__()}")
    for i in range(context_dataset.__len__()):
        context_dataset.__getitem__(i)
    d = context_dataset.__getitem__(i)

    for key in d["target"].keys():
        # pring shape of each key
        try:
            print(f"{key}: {d['target'][key].shape}")
        except:
            pass
    if mode == "test":
        # Test should stard from "2023-01-24 00:00:00"
        d = context_dataset.__getitem__(seq_len)
        print(
            reconstruct_datetime_from_features(
                d["target"]["x_mark"].long().numpy(),
                ["year", "month", "day", "weekday", "hour"],
            )[:2]
        )

ContextBatchDataset train size: 37921
x: torch.Size([48, 1])
y: torch.Size([48, 1])
x_mark: torch.Size([48, 5])
y_mark: torch.Size([48, 5])
ContextBatchDataset val size: 4201
x: torch.Size([48, 1])
y: torch.Size([48, 1])
x_mark: torch.Size([48, 5])
y_mark: torch.Size([48, 5])
ContextBatchDataset test size: 8185
x: torch.Size([48, 1])
y: torch.Size([48, 1])
x_mark: torch.Size([48, 5])
y_mark: torch.Size([48, 5])
0   2023-01-24 00:00:00
1   2023-01-24 01:00:00
dtype: datetime64[ns]


In [4]:
print((d['target']['x'].shape))
print((d['target']['y'].shape))

torch.Size([48, 1])
torch.Size([48, 1])


In [5]:
print((d['context']['x'].shape))
print((d['context']['y'].shape))

torch.Size([48, 52])
torch.Size([48, 52])


In [6]:
print((d['target']['x_mark'] == d['context']['x_mark']).all())
print((d['target']['y_mark'] == d['context']['y_mark']).all())

tensor(True)
tensor(True)


In [11]:
from lmetk.data.datamodules import ContextBaseDataModule


train_dataset = ContextBatchDataset(
        features=splitted_data['train'],
        time_features=splitted_timestamps['train'],
        seq_len=seq_len,
        label_len=label_len,
        pred_len=pred_len,
        mask=dataset_loader2.data_dict["mask"],
        stride=24,
    )


val_dataset = ContextBatchDataset(
        features=splitted_data['val'],
        time_features=splitted_timestamps['val'],
        seq_len=seq_len,
        label_len=label_len,
        pred_len=pred_len,
        mask=dataset_loader2.data_dict["mask"],
        stride=24,
    )


test_dataset = ContextBatchDataset(
        features=splitted_data['test'],
        time_features=splitted_timestamps['test'],
        seq_len=seq_len,
        label_len=label_len,
        pred_len=pred_len,
        mask=dataset_loader2.data_dict["mask"],
        stride=24,
    )

datamodule = ContextBaseDataModule(
    dataset_train=train_dataset, 
    dataset_val=val_dataset, 
    dataset_test=test_dataset,

    train_batch_size=1,
    val_batch_size=1,
    test_batch_size=1,

    shuffle_train=False,
    shuffle_val=False,
    shuffle_test=False,
    )

In [12]:
batch = next(iter(datamodule.train_dataloader()))

In [13]:
datamodule.train_dataloader().__len__()

1581

In [14]:
import torch
def dl_to_sklearn__with_context_for_tabpfn(dl, lag=0):
    X, y = [], []
    for batch in iter(dl):
        x_target, y_target, x_t_time, y_t_time = batch['target']['x'], batch['target']['y'], batch['target']['x_mark'], batch['target']['y_mark']
        x_context, y_context, x_c_time, y_c_time = batch['context']['x'], batch['context']['y'], batch['context']['x_mark'], batch['context']['y_mark']
        
        y_context, y_c_time, x_target, x_t_time

        X.append(torch.concat([x_context[:, lag, :], y_target.squeeze(2)], dim=1))
        y.append(y_t[:, lag, :])

    return torch.concat(X).numpy(), torch.concat(y).numpy()

In [15]:
dl_to_sklearn__with_context_for_tabpfn(datamodule.train_dataloader(), lag=0)

NameError: name 'y_t' is not defined

In [ ]:
x_target, y_target, x_t_time, y_t_time = batch['target']['x'], batch['target']['y'], batch['target']['x_mark'], batch['target']['y_mark']
x_context, y_context, x_c_time, y_c_time = batch['context']['x'], batch['context']['y'], batch['context']['x_mark'], batch['context']['y_mark']

concat = torch.concat([x_context, x_t_time, x_target], dim=-1)

In [87]:
concat.shape

torch.Size([1, 48, 58])

In [ ]:
from lmetk.case_studies.railway_traction import SpatialDataModule

dm = SpatialDataModule(ts_in=24, batch_size=1)
dm.setup()


In [ ]:
dm.datasets['train']

TypeError: iteration over a 0-d tensor

In [82]:
def dl_to_sklearn__with_context_for_tabpfn_original(dl, lag=0):
    X, y = [], []
    for batch in iter(dl):
        x_c, y_c, x_t, y_t = batch
        X.append(torch.concat([x_t[:, lag, :], y_c.squeeze(2)], dim=1))
        y.append(y_t[:, lag, :])

    return torch.concat(X).numpy(), torch.concat(y).numpy()

X, y = dl_to_sklearn__with_context_for_tabpfn_original(dm.train_dataloader())

In [68]:
X.shape

(39193, 80)